In [4]:
import pandas as pd
import requests
import mysql.connector
from mysql.connector import Error


DB_CONFIG = {
    "host": "localhost",
    "user": "root",   
    "password": "Nishan@20640904",  
    "database": "etl_project"
}


def extract():

    url = "https://jsonplaceholder.typicode.com/users"

    try:
        response = requests.get(url)
        response.raise_for_status()
        data = response.json()
        data=pd.json_normalize(data)
        print(f"Successfully extracted {len(data)} rows")
        return pd.DataFrame(data)

    except requests.exceptions.RequestException as e:
        print("API Request Failed:", e)
        return pd.DataFrame()


def transform(df):
    print("\nStarting TRANSFORM step...")

    if df.empty:
        print("DataFrame is empty")
        return df

    users_df = pd.DataFrame()
    users_df["id"] = df["id"]
    users_df["name"] = df["name"].astype(str).str.strip()
    users_df["email"] = df["email"].astype(str).str.lower()
    users_df["username"] = df["username"]
    users_df["city"] = df["address.city"].astype(str).str.strip()
    

    users_df.dropna(inplace=True)

    users_df.drop_duplicates(subset=["id"], inplace=True)

    users_df["name_length"] = users_df["name"].apply(len)

    users_df["email_domain"] = users_df["email"].apply(
        lambda x: x.split("@")[1]
    )

    users_df["user_category"] = users_df["name_length"].apply(
        lambda x: "Long Name" if x > 12 else "Short Name"
    )

    print("Transformation completed")
    print(f"Rows after cleaning: {len(users_df)}")

    return users_df


def load(df):

    if df.empty:
        print("No data to load")
        return

    try:
        # Connect to MySQL
        connection = mysql.connector.connect(**DB_CONFIG)

        if connection.is_connected():

            cursor = connection.cursor()

            # ---------------------------------
            # Create table if not exists
            # ---------------------------------

            create_table_query = """
            CREATE TABLE IF NOT EXISTS users (
                id INT PRIMARY KEY,
                name VARCHAR(255),
                email VARCHAR(255),
                username VARCHAR(255),
                city VARCHAR(255),
                name_length INT,
                email_domain VARCHAR(255),
                user_category VARCHAR(100)
            )
            """

            cursor.execute(create_table_query)

            print("Table checked/created")

            insert_query = """
            INSERT INTO users (
                id,
                name,
                email,
                username,
                city,
                name_length,
                email_domain,
                user_category
            )
            VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
            ON DUPLICATE KEY UPDATE
                name = VALUES(name),
                email = VALUES(email),
                username = VALUES(username),
                city = VALUES(city),
                name_length = VALUES(name_length),
                email_domain = VALUES(email_domain),
                user_category = VALUES(user_category)
            """

            data_to_insert = []

            for _, row in df.iterrows():
                data_to_insert.append((
                    int(row["id"]),
                    row["name"],
                    row["email"],
                    row["username"],
                    row["city"],
                    int(row["name_length"]),
                    row["email_domain"],
                    row["user_category"]
                ))

            cursor.executemany(insert_query, data_to_insert)

            connection.commit()

            print(f"{cursor.rowcount} rows inserted/updated")


            df.to_csv("cleaned_users.csv", index=False)

            print("CSV exported successfully")

    except Error as e:
        print("MySQL Error:", e)

    finally:
        if 'connection' in locals() and connection.is_connected():
            cursor.close()
            connection.close()
            print("MySQL connection closed")



def run_pipeline():

    print("=" * 50)
    print("RUNNING FULL ETL PIPELINE")
    print("=" * 50)

    # EXTRACT
    raw_data = extract()

    # TRANSFORM
    clean_data = transform(raw_data)

    # LOAD
    load(clean_data)

    print("\nETL Pipeline Completed Successfully")


# =========================================
# RUN PROGRAM
# =========================================

if __name__ == "__main__":
    run_pipeline()

RUNNING FULL ETL PIPELINE
Successfully extracted 10 rows

Starting TRANSFORM step...
Transformation completed
Rows after cleaning: 10
Table checked/created
10 rows inserted/updated
CSV exported successfully
MySQL connection closed

ETL Pipeline Completed Successfully
